# GEOparse로 다른 암종도 분석해보기
## 개요
우리 전공을 하다 보면 제일 많이 다루는게 암, 알츠하이머, 파킨슨입니다. 그 중에서도 암 하면서 제일 많이 나오는 변이원이 담배인데... 아주 담배는 진짜 심심하면 나와요... 교수님들도 담배 피지 말라고 하시는데... 그럼에도 피는 분이 계세요. 수업중에 헉하면서도... 그죠. 니코틴이 무섭습니다. 

그래서 자주 걸리는 암종을 묶어서 분석해보려고 합니다. 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, GEOparse(NCBI GEO에 접근할 때 필요)
- 데이터 리소스: GEOparse

## 그래프 세팅
- 폰트: 갈무리11(일반 분석)/나눔스퀘어(포폴용)
- 폰트사이즈: 제목 16, 축 라벨 12
- color scheme
    1. 기반색: cloud dancer(#f0f9fc)
    2. 그래프용 팔레트
        - 비흡연자: #FDD835
        - 흡연자: #182583
        - 기타: #95a5a6 (Ash gray)
    3. cmap(단색 그라데이션)
        - 비흡연자: #FDD835
        - 흡연자: #182583
- 배경: whitegrid

In [ ]:
# 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse # NCBI GEO에 접근할 때 필요함 

from Bio import Entrez # NCBI 창고털이 드가자 

# 통계분석용
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Galmuri11'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False

# 컬러 지정
cloud_Dancer = '#f0f9fc'
blue = '#0077b6'
yellow = '#fdd835'
gray = '#95a5a6'

# 컬러테이블
custom_yellow = sns.blend_palette([cloud_Dancer,yellow], as_cmap=True)
custom_blue = sns.blend_palette([cloud_Dancer,blue], as_cmap=True)
custom_gray = sns.blend_palette([cloud_Dancer,gray], as_cmap=True)

# 확인하고 가세용
display(custom_yellow)
display(custom_blue)
display(custom_gray)

# NCBI 창고를 털려면 이메일이 필요함 
Entrez.email = "blackholekun@gmail.com" # 이메일 

# 데이터 가져오기
- GEO에서 암종별로 데이터를 가져올겁니다. 가능하면 흡연/비흡연 여부 뿐 아니라 다른 인자도 볼 수 있으면 보겠습니다. (예: 가족력)
- 오래 걸리니까 돌려놓고 똥 함 누고 오십셔. 

## 신장암
- 아니 위암 달랬더니 신장암은 왜 찾아온거니 제미나이야... 

In [ ]:
# GSE46699 데이터 받아오기
gse = GEOparse.get_GEO(geo="GSE46699", destdir="./")

metadata_df = gse.phenotype_data
print(f"전체 샘플 수: {len(metadata_df)}")
print(metadata_df.head())

In [ ]:
# 내놓아라 메타데이터
first_gsm = list(gse.gsms.values())[0]
print("--- 샘플 메타데이터 확인 ---")
for v in first_gsm.metadata['characteristics_ch1']:
    print(v)

- 비만에서 급 뼈맞고 갑니다. 
- 분석 그룹: 
    1. 노말
    2. 뚠뚠(비만)
    3. 구름과자(=담배)
    4. 뚠뚠이+구름과자(비만+흡연) 

In [ ]:
meta_df = gse.phenotype_data

def classify_smoking_obese_v2(row):
    # 'characteristics_ch1.2.smoking'과 'characteristics_ch1.3.obese' 컬럼을 사용
    smk = str(row.get('characteristics_ch1.2.smoking', 'no')).lower()
    obs = str(row.get('characteristics_ch1.3.obese', 'no')).lower()
    
    if 'yes' in smk:
        return '4. 흡연+비만' if 'yes' in obs else '3. 흡연'
    else:
        return '2. 금연+비만' if 'yes' in obs else '1. 금연'

meta_df['Analysis_Group'] = meta_df.apply(classify_smoking_obese_v2, axis=1)

# 그룹별 샘플 수 확인
print("--- 분석 그룹 분포 ---")
print(meta_df['Analysis_Group'].value_counts().sort_index())

In [ ]:
# 1. StopIteration 피하기: 수동으로 발현량 데이터프레임(exp_df) 생성
# gse.gsms 내부의 데이터를 직접 딕셔너리로 변환하여 데이터프레임을 만듭니다.
all_samples = {}
for gsm_name, gsm in gse.gsms.items():
    # 각 샘플의 'ID_REF'를 인덱스로, 'VALUE'를 값으로 가져옵니다.
    all_samples[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

exp_df = pd.DataFrame(all_samples)

# 2. 첫 번째 유전자 데이터 추출 (예: '1007_s_at')
target_gene_id = exp_df.index[0]
# 수치형으로 변환 (Series 형태)
gene_series = pd.to_numeric(exp_df.loc[target_gene_id], errors='coerce')

# 3. 메타데이터와 수치 결합
plot_df = meta_df.copy()
plot_df['Expression'] = gene_series

# 4. 그룹별 데이터 리스트 생성 (Matplotlib boxplot용)
# Analysis_Group 순서대로 데이터를 모읍니다.
group_list = sorted(plot_df['Analysis_Group'].unique())
data_to_plot = [plot_df[plot_df['Analysis_Group'] == g]['Expression'].dropna().values for g in group_list]

# 5. 컬러 설정 (사용자님 커스텀 팰럿)
# Group 2(사용자님)는 2번째 위치
colors = [custom_gray(0.4), custom_blue(0.5), custom_blue(0.9), custom_yellow(0.9)]

# 6. Matplotlib으로 그리기
fig, ax = plt.subplots(figsize=(10, 6))

# 박스플롯 생성
bp = ax.boxplot(data_to_plot, patch_artist=True, widths=0.5)

# 박스 색상 및 테두리 수동 설정
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(colors[i])
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)

# 중앙값 선(Median) 강조
for median in bp['medians']:
    median.set(color='firebrick', linewidth=2)

# 7. "Me!" 주석 (사용자님은 2번 그룹)
me_idx = 1 # 리스트 상의 2번째
me_median = np.median(data_to_plot[me_idx]) if len(data_to_plot[me_idx]) > 0 else 0
ax.annotate('Me!', 
            xy=(me_idx + 1, me_median), 
            xytext=(me_idx + 1.3, max(plot_df['Expression'])),
            fontsize=12, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# 축 및 타이틀 설정
ax.set_title(f"유전자 {target_gene_id}: 비만과 흡연의 영향", fontsize=15, pad=15)
ax.set_xticklabels(group_list)
ax.set_ylabel("발현 수치(Expression)")

plt.tight_layout()
plt.show()

- 쇠질 하라고 한 이유가 있었군. 

### ANOVA
- 네 그룹간에 통계적으로 유의한 차이가 있는지 확인할 예정

In [ ]:
# 1. 데이터 추출 및 수치화
# 1007_s_at 대신, 이번에는 차이가 좀 있을법한 다른 유전자를 한번 보시죠.
# 예시로 10번째 인덱스에 있는 유전자를 가져와보겠습니다.
target_idx = 10 
target_gene = exp_df.index[target_idx]

plot_df = meta_df.copy()
plot_df['Expression'] = pd.to_numeric(exp_df.iloc[target_idx], errors='coerce')

# 2. 그룹별 분리
g1 = plot_df[plot_df['Analysis_Group'].str.contains('1')]['Expression'].dropna()
g2 = plot_df[plot_df['Analysis_Group'].str.contains('2')]['Expression'].dropna()
g3 = plot_df[plot_df['Analysis_Group'].str.contains('3')]['Expression'].dropna()
g4 = plot_df[plot_df['Analysis_Group'].str.contains('4')]['Expression'].dropna()

# 3. ANOVA 실행 (정규성 만족 시 가장 강력한 도구)
f_stat, p_val_anova = stats.f_oneway(g1, g2, g3, g4)

print(f"--- ANOVA 분석 결과 ({target_gene}) ---")
print(f"F-통계량: {f_stat:.4f}")
print(f"ANOVA P-value: {p_val_anova:.4f}")

# 4. 결과 해석 및 시각화
if p_val_anova < 0.05:
    print(f"✅ 유의미한 차이 발견! (P < 0.05)")
else:
    print(f"❌ 이 유전자({target_gene})도 모든 그룹에서 비슷하게 발현되네요.")

### ANOVA 친구 튜키

In [ ]:
tukey = pairwise_tukeyhsd(endog=plot_df['Expression'], groups=plot_df['Analysis_Group'], alpha=0.05)
    
# 결과를 데이터프레임으로 예쁘게 보기
tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
print(tukey_df)

# 4. 시각화 
fig = tukey.plot_simultaneous(figsize=(10, 6))
plt.title(f"Tukey HSD 그룹별 차이 비교: {exp_df.index[target_idx]}", fontsize=15)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

#### 너 뭐냐?

In [ ]:
# 1. 이 데이터셋이 사용하는 GPL ID 확인
gpl_id = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_id]

# 2. 1438_at의 정보 추출
# gpl.table에서 'ID'가 '1438_at'인 행을 찾습니다.
probe_info = gpl.table[gpl.table['ID'] == '1438_at']

print(f"--- [플랫폼 정보: {gpl_id}] ---")
if not probe_info.empty:
    # 보통 'Gene Symbol' 혹은 'Symbol' 컬럼에 이름이 있습니다.
    symbol = probe_info['Gene Symbol'].values[0] if 'Gene Symbol' in gpl.table.columns else "N/A"
    description = probe_info['Gene Title'].values[0] if 'Gene Title' in gpl.table.columns else "N/A"
    
    print(f"유전자 기호(Symbol): {symbol}")
    print(f"유전자 설명: {description}")
else:
    print("해당 Probe ID를 찾을 수 없습니다. 인덱스를 다시 확인해 주세요.")

- EPHB3: Ephrin type-B receptor 3
- 보통 RTK는 변이 터지면 엑셀되던데 쟤는 특이하네. 

#### TOP 10 Significant gene

In [ ]:
# 전체 유전자 ANOVA 스캐닝 (가장 차이가 큰 녀석 찾기)
results = []
for i in range(len(exp_df)):
    series = pd.to_numeric(exp_df.iloc[i], errors='coerce')
    temp_df = meta_df.copy()
    temp_df['Exp'] = series
    
    gs = [temp_df[temp_df['Analysis_Group'] == g]['Exp'].dropna() for g in sorted(temp_df['Analysis_Group'].unique())]
    f, p = stats.f_oneway(*gs)
    results.append((exp_df.index[i], p))

# P-value 순으로 정렬하여 상위 10개 출력
significant_genes = pd.DataFrame(results, columns=['Gene_ID', 'p_value']).sort_values('p_value')
print("--- [Top 10 Significant Genes] ---")
print(significant_genes.head(10))

In [ ]:
# 1위 유전자 정보 추출
top_gene_id = "207435_s_at"
top_gene_info = gpl.table[gpl.table['ID'] == top_gene_id]

if not top_gene_info.empty:
    symbol = top_gene_info['Gene Symbol'].values[0]
    title = top_gene_info['Gene Title'].values[0]
    print(f"--- [최종 분석 보고서] ---")
    print(f"압도적 1위 유전자: {symbol}")
    print(f"역할: {title}")
    
    # 2. 이 녀석으로 마지막 튜키(Tukey) 한 번만 더!
    plot_df = meta_df.copy()
    plot_df['Expression'] = pd.to_numeric(exp_df.loc[top_gene_id], errors='coerce')
    
    tukey = pairwise_tukeyhsd(endog=plot_df['Expression'], 
                            groups=plot_df['Analysis_Group'], 
                            alpha=0.05)
    print("\n[Tukey HSD Summary]")
    print(tukey.summary())

    fig = tukey.plot_simultaneous(figsize=(10, 6))
    plt.title(f"Tukey HSD 그룹별 차이 비교: {symbol}", fontsize=15)
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.show()
else:
    print("정보를 찾을 수 없습니다.")



- 얘는 또 골때리는게 그냥 비만하기만 한 사람들은 정상 체중(금연)에 비해 오히려 발현률이 낮다. 
- 근데 또 담배 피는 사람들은 정상체중(금연)보다 발현률이 높다 이거죠. 저 친구 걍 스플라이싱 해 주는 앤데. 
- 활성산소나 나트륨처럼 어느정도는 발현이 되어야 하는 유전자가 아닐까? 

## 대장암

In [ ]:
# GSE39582 데이터 받아오기
gse = GEOparse.get_GEO(geo="GSE39582", destdir="./")

metadata_df = gse.phenotype_data
print(f"전체 샘플 수: {len(metadata_df)}")
print(metadata_df.head())

In [ ]:
# 'characteristics'가 포함된 컬럼들만 리스트업해서 내용을 슬쩍 봅니다.
char_cols = [c for c in gse.phenotype_data.columns if 'characteristics' in c]
print("--- [임상 정보 의심 컬럼들] ---")
print(gse.phenotype_data[char_cols].head(3))

In [ ]:
# 1. 비교 그룹 설정 (Stage 1 vs Stage 4)
df_stage = gse.phenotype_data[['characteristics_ch1.4.tnm.stage']].copy()
df_stage['Stage'] = df_stage['characteristics_ch1.4.tnm.stage']

s1_samples = df_stage[df_stage['Stage'] == '1'].index
s4_samples = df_stage[df_stage['Stage'] == '4'].index

print(f"Stage 1 샘플 수: {len(s1_samples)}")
print(f"Stage 4 샘플 수: {len(s4_samples)}")

exp_crc = pd.DataFrame({gsm_name: gsm.table.set_index('ID_REF')['VALUE'] 
                        for gsm_name, gsm in gse.gsms.items()})

# 2. 전수 조사 (상위 54,675개 유전자 대상)
res = []
for probe_id in exp_crc.index:
    group_s1 = pd.to_numeric(exp_crc.loc[probe_id, s1_samples], errors='coerce').dropna()
    group_s4 = pd.to_numeric(exp_crc.loc[probe_id, s4_samples], errors='coerce').dropna()
    
    if len(group_s1) > 5 and len(group_s4) > 5:
        t_stat, p_val = stats.ttest_ind(group_s1, group_s4)
        res.append((probe_id, p_val, group_s4.mean() - group_s1.mean()))

# 3. 결과 정리
top_culprits = pd.DataFrame(res, columns=['Probe_ID', 'P_value', 'Diff']).sort_values('P_value')
print("\n--- [대장암 악화의 메이저 주범 TOP 5] ---")
print(top_culprits.head(5))

In [ ]:
# 1. top_culprits 상위 5개를 복사하고 컬럼명을 'ID'로 통일
top_5_results = top_culprits.head(5).copy()
top_5_results = top_5_results.rename(columns={'Probe_ID': 'ID'})

# 2. top_5_results에 있는 ID들만 추출해서 리스트로 만듦 (자동화)
actual_ids = top_5_results['ID'].tolist()

# 3. GPL 테이블에서 해당 ID들의 정보 추출
target_info = gpl.table[gpl.table['ID'].isin(actual_ids)][['ID', 'Gene Symbol', 'Gene Title']]

# 4. 최종 병합 및 출력
final_view = pd.merge(top_5_results, target_info, on='ID')

print("--- [대장암 4기 빌런 & 수호자 정체 최종 확인] ---")
# 가독성을 위해 출력 컬럼 순서 조정
print(final_view[['ID', 'Gene Symbol', 'Diff', 'P_value', 'Gene Title']])

1. PCAT6: Prostate Cancer Associated Transcript 6
2. SLC35G1: Solute Carrier Family 35 Member 1
3. CASP1: Caspase 1
4. VWA3A: Von Willebrand Factor A Domain Containing 3A
5. ULK4: Unc-51 Like Kinase 4

## 유방암

In [ ]:
# 1. 유방암 데이터셋 로드 (GSE20685)
gse_breast = GEOparse.get_GEO(geo="GSE20685", destdir="./")

# 2. 메타데이터 컬럼 확인
# 유방암은 ER(에스트로겐 수용체), PR, HER2 상태가 중요합니다.
print("--- [GSE20685 유방암 임상 정보 컬럼] ---")
print(gse_breast.phenotype_data.columns)

# 3. 발현량 데이터프레임 생성
exp_breast = pd.DataFrame({gsm_name: gsm.table.set_index('ID_REF')['VALUE'] 
                        for gsm_name, gsm in gse_breast.gsms.items()})

In [ ]:
# 1. 전이 여부 컬럼 정리
meta_breast = gse_breast.phenotype_data
meta_breast['Metastasis'] = meta_breast['characteristics_ch1.5.event_metastasis']

# 전이 있음(1) vs 없음(0) 샘플 분리
m1_samples = meta_breast[meta_breast['Metastasis'] == '1'].index
m0_samples = meta_breast[meta_breast['Metastasis'] == '0'].index

print(f"분석 대상: 전이 발생({len(m1_samples)}개) vs 전이 없음({len(m0_samples)}개)")

# 2. 전이 주범 유전자 전수 조사
results_brca = []
for probe in exp_breast.index:
    g1 = pd.to_numeric(exp_breast.loc[probe, m1_samples], errors='coerce').dropna()
    g0 = pd.to_numeric(exp_breast.loc[probe, m0_samples], errors='coerce').dropna()
    
    if len(g1) > 5 and len(g0) > 5:
        _, p = stats.ttest_ind(g1, g0)
        results_brca.append((probe, p, g1.mean() - g0.mean()))

# 3. TOP 5 빌런 추출
top_brca_culprits = pd.DataFrame(results_brca, columns=['Probe_ID', 'P_value', 'Diff']).sort_values('P_value').head(5)

# 4. 정체 확인 (GPL 정보 병합)
top_brca_info = pd.merge(top_brca_culprits.rename(columns={'Probe_ID': 'ID'}), 
                        gpl.table[['ID', 'Gene Symbol', 'Gene Title']], on='ID')

print("\n--- [유방암 전이 주범: 치명적 빌런 TOP 5] ---")
print(top_brca_info[['ID', 'Gene Symbol', 'Diff', 'P_value', 'Gene Title']])

- 얘는 전이 주범을 찾은거고, 이번에는 정상 조직과 종양 조직을 비교해보자. 

In [ ]:
# 1. 정상/암 대조 데이터셋 로드 (GSE15852)
gse_comp = GEOparse.get_GEO(geo="GSE15852", destdir="./")

# 2. 샘플 분류 (정상 vs 암)
# 이 데이터셋은 Title에 'Normal'과 'Tumor'가 명시되어 있습니다.
meta_comp = gse_comp.phenotype_data
normal_idx = meta_comp[meta_comp['title'].str.contains('Normal', na=False)].index
tumor_idx = meta_comp[meta_comp['title'].str.contains('Tumor', na=False)].index

print(f"비교군: 정상 조직({len(normal_idx)}개) vs 암 조직({len(tumor_idx)}개)")

# 3. 우리가 찾은 빌런들의 수치 비교
target_genes = ["213375_s_at", "218553_s_at"] # N4BP2L1, KCTD15
exp_comp = pd.DataFrame({gsm_name: gsm.table.set_index('ID_REF')['VALUE'] 
                        for gsm_name, gsm in gse_comp.gsms.items()})

for probe in target_genes:
    norm_val = pd.to_numeric(exp_comp.loc[probe, normal_idx]).mean()
    tumor_val = pd.to_numeric(exp_comp.loc[probe, tumor_idx]).mean()
    diff = tumor_val - norm_val
    print(f"--- [ID: {probe}] ---")
    print(f"정상 평균: {norm_val:.2f} / 암 평균: {tumor_val:.2f}")
    print(f"차이: {diff:.2f} ({'상승!!' if diff > 0 else '하락!!'})")

In [ ]:
# 1. title 컬럼을 기준으로 샘플 분리
normal_idx = meta_comp[meta_comp['title'].str.contains('Normal', na=False)].index
cancer_idx = meta_comp[meta_comp['title'].str.contains('Cancer', na=False)].index

print(f"🎯 매칭 성공! 정상군({len(normal_idx)}개) vs 암환자군({len(cancer_idx)}개)")

# 2. 우리의 타겟 유전자 비교
target_probes = ["213375_s_at", "218553_s_at"] # N4BP2L1, KCTD15

print("\n--- [유방암: 정상 vs 암 진짜 격차 보고서] ---")
for probe in target_probes:
    n_vals = pd.to_numeric(exp_comp.loc[probe, normal_idx], errors='coerce').dropna()
    c_vals = pd.to_numeric(exp_comp.loc[probe, cancer_idx], errors='coerce').dropna()
    
    n_mean = n_vals.mean()
    c_mean = c_vals.mean()
    diff = c_mean - n_mean
    
    symbol = gpl.table[gpl.table['ID'] == probe]['Gene Symbol'].values[0]
    
    print(f"▶ 유전자: {symbol}")
    print(f"   [정상 상태]: {n_mean:.2f}")
    print(f"   [암세포 상태]: {c_mean:.2f}")
    print(f"   [격차]: {diff:+.2f} ({'⚠️ 암이 에너지를 빨아먹는 중' if diff > 0 else '🛡️ 방어막 파괴됨'})")
    print("-" * 35)

- N4BP2L1: NEDD4 Binding Protein 2 Like 1
- KCTD15: Potassium Channel Tetramerization Domain Containing 15
- BRCA가 범인일 줄 알았는데 걔는 유전되는 친구라 후천적으로 발현되고 그러지는 않는다고 한다. 